In [12]:
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

import math
from collections import Counter

In [8]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ESHAAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ESHAAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [ ]:
class CaptionTokeniser :
    def __init__(self,  max_vocab_size = None):
        self.max_vocab_size = max_vocab_size
        
        self.word_to_idx = {
            "<PAD>": 0,
            "<BOS>": 1,
            "<EOS>": 2,
            "<UNK>": 3
        }


        self.vocab = self.word_to_idx
        self.idx_to_word = {
            0 : "<PAD>",
            1 : '<UNK>'
        }

    def create_vocab(self,text):
        tokens = word_tokenize(text.lower())
        tokens = [
            word for word in tokens
            if word.isalpha()
        ]
        word_counts = Counter(tokens)
        for word, count in word_counts.most_common():
            if count >= 2:
                self.word_to_idx[word] = len(self.word_to_idx)
                self.idx_to_word[len(self.word_to_idx)] = word


    def add_to_vocab(self,text):
        words = [word for word in word_tokenize(text.lower())
                 if word.isalpha()]
        candidate_vocab = set(words)
        for word in candidate_vocab :
            if word not in self.vocab.keys():
                self.word_to_idx[word] = len(self.vocab.keys())
                self.idx_to_word[len(self.word_to_idx.keys())] = word


    def encode(self, text):
        tokens = word_tokenize(text.lower())
        out = [self.word_to_idx[word] for word in tokens]
        return out

    def decode(self, idx_arr):
        out = [self.idx_to_word[idx] for idx in idx_arr]
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,embed_dim, num_heads, masked = False):
        super().__init__()
        if embed_dim%num_heads != 0 :
            raise ValueError(f"cant divide embed_dim{embed_dim} into {num_heads} heads")

        self.masked = masked
        self.num_heads = num_heads
        self.q = nn.Linear(embed_dim,embed_dim)
        self.k = nn.Linear(embed_dim,embed_dim)
        self.v = nn.Linear(embed_dim,embed_dim)
        self.Wo = nn.Linear(embed_dim,embed_dim)


    def forward(self, input_batch): 
        B, T, E = input_batch.shape
        device = input_batch.device
        H = self.num_heads
        Eh = E//self.num_heads

        query_vec = self.q(input_batch)  # shape (B,T,E)x(ExE) = BxTxE
        key_vec = self.k(input_batch)    # shape (B,T,E)x(ExE) = BxTxE
        value_vec = self.v(input_batch)  # shape (B,T,E)x(ExE) = BxTxE

        head_q_vecs = query_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh) 
        head_k_vecs = key_vec.reshape(B,T,H,Eh).transpose(1,2)      #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)
        head_v_vecs = value_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)

        sim_scores = head_q_vecs @ head_k_vecs.transpose(-2,- 1) # (B,H,T,Eh) . (B,H,Eh,T) = (B,H,T,T)
        sim_scores = sim_scores/math.sqrt(Eh) #(B,H,T,T)

        if self.masked :
            t_q = torch.arange(T, device=device).view(T,1)
            t_k = torch.arange(T, device=device).view(1,T)
            mask_0 = t_q >= t_k
            mask_inf = t_q < t_k 
            sim_scores = sim_scores.masked_fil(
                mask_inf,
                float('-inf')
            )         
        sim_scores = torch.softmax(sim_scores, dim = -1)#(B,H,Tq,Tk) , dim =1 , as we are softmaxing ALL KEY probs for a QUERY


        attention = sim_scores @ head_v_vecs # (B,H,T,T).(B,H,T,Eh) = (B,H,T,Eh)
        out = attention.transpose(1,2).reshape(B,T,E) # from trnaspose : (B,T,H,Eh), from reshape : (B,T,E)
        output = self.Wo(out) #(B,T,E)x(E,E) = (B,T,E)
        return output #(B,T,E)
        

In [ ]:
class ResidualConnect(nn.Module):
    def __init__(self, sublayer):
        super().__init__()
        self.sublayer = sublayer
        
    def forward(self,x):
        return x + self.sublayer(x)

In [ ]:
def positionalEncoder(tensor):
    B,T,E = tensor.shape
    pos_encods = torch.zeros((1,T,E), device=tensor.device)
    for t in range(T) :
        for e in range(0,E,2) :
            pos_encods[:,t,e] = math.sin(t/math.pow(10000, e/E))
            pos_encods[:,t,e+1] = math.cos(t/math.pow(10000, e/E))
    return pos_encods


class AddPositionalEmbeds(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, tensor):
        B,T,E = tensor.shape
        pos_encods = positionalEncoder(tensor)
        return tensor + pos_encods
        
        

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, attention_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.MaskedAttention = MultiHeadAttention(embed_dim, attention_heads)
        self.fnn = nn.Sequential(
            nn.Linear(embed_dim, 2048),
            nn.ReLU(),
            nn.Linear(2048,embed_dim)
        )
        self.layerNorm1 = nn.LayerNorm(self.embed_dim)
        self.layerNorm2 = nn.LayerNorm(self.embed_dim)

    def forward(self, input_embeds, image_features): 
        o = input_embeds + self.MaskedAttention(input_embeds)
        o = self.layerNorm1(o)
        o = o + self.fnn(o)
        o = self.layerNorm2(o)
        return o

In [ ]:
class SimpleDecoder(nn.Module):
    def __init__(self, vocab_count, embedding_dim,attention_heads, n_blocks):
        super().__init__()
        self.n_blocks = n_blocks
        self.embedding = nn.Embedding(vocab_count, embedding_dim)
        self.add_positional = AddPositionalEmbeds()
        self.onn = nn.Sequential(
                    nn.Linear(embedding_dim, vocab_count),
        )

        self.decoders = nn.ModuleList(DecoderBlock(embedding_dim, attention_heads) for _ in range(n_blocks))

        

    def forward(self,batch):
        return